# FPS Point Cloud Downsampling and Upsampling

Downsample points with FPS, gather anchor features, then upsample features back to the original points.

## Single Sample

In [ ]:
import torch
import torchbvh as tb

N = 100000
M = int(N * 0.25) # 25% downsample ratio
points = torch.randn(N, 3, device="cuda") # (x, y, z) coordinates of points
x = torch.randn(N, 8, device="cuda")  # per-point features

# downsample to M points
downsampled = tb.fps(points, target_tokens=M)
downsampled_points = downsampled.points

# Upsample back to original N points by assigning each point its nearest anchor feature
downsampled_x = x[downsampled.indices]  # (M, C)
upsampled_x = downsampled_x[downsampled.nearest_anchor]  # (N, C)

print("input features:", x.shape)
print("downsampled points:", downsampled_points.shape)
print("downsampled features:", downsampled_x.shape)
print("original/upsampled points:", points.shape)
print("upsampled features:", upsampled_x.shape)

## Batched

In [ ]:
import torch
import torchbvh as tb

# points: (B, N, D), x: (B, N, C)
B, N, D, C = 16, 10000, 3, 8
M = int(N * 0.25)
points = torch.randn(B, N, D, device="cuda")
x = torch.randn(B, N, C, device="cuda")

downsampled = tb.fps(points, target_tokens=M)

downsampled_x = torch.gather(
    x,
    1,
    downsampled.indices[:, :, None].expand(-1, -1, x.size(-1)),
)  # (B, M, C)

upsampled_x = torch.gather(
    downsampled_x,
    1,
    downsampled.nearest_anchor[:, :, None].expand(-1, -1, x.size(-1)),
)  # (B, N, C)

print("input features:", x.shape)
print("downsampled points:", downsampled.points.shape)
print("downsampled features:", downsampled_x.shape)
print("original/upsampled points:", points.shape)
print("upsampled features:", upsampled_x.shape)